<a href="https://colab.research.google.com/github/dikaproject126/UTS-PDS/blob/main/StudiKasus/Kelompok_4_Soal_4_Pemodelan_dan_Simulasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import itertools
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linprog

#data model
c = np.array([12_000_000, 10_000_000], dtype=float)
A = np.array([[4, 2],
              [2, 3]], dtype=float)
b = np.array([40, 36], dtype=float)


#1) metode grafik (titik sudut)
def corner_points(A, b):
    """Semua perpotongan garis kendala (termasuk sumbu), lalu saring yang layak."""
    lines = [(a, bi) for a, bi in zip(A, b)]
    lines += [(np.array([1.0, 0.0]), 0.0), (np.array([0.0, 1.0]), 0.0)]  # x1=0, x2=0
    pts = []
    for (a1, b1), (a2, b2) in itertools.combinations(lines, 2):
        M = np.array([a1, a2])
        if abs(np.linalg.det(M)) < 1e-12:
            continue
        p = np.linalg.solve(M, np.array([b1, b2]))
        if np.all(p >= -1e-9) and np.all(A @ p <= b + 1e-9):
            pts.append(tuple(np.round(p, 6)))
    return sorted(set(pts))


#2) simpleks manual
def simplex_max(c, A, b, verbose=True):
    m, n = A.shape
    T = np.zeros((m + 1, n + m + 1))
    T[:m, :n] = A
    T[:m, n:n + m] = np.eye(m)
    T[:m, -1] = b
    T[m, :n] = -c
    basis = [n + i for i in range(m)]
    names = [f"x{i+1}" for i in range(n)] + [f"s{i+1}" for i in range(m)]
    it = 0
    while True:
        if verbose:
            print(f"\n--- Tabel simpleks iterasi {it} ---")
            print("Basis |", " ".join(f"{nm:>14}" for nm in names), "|          RHS")
            for i in range(m):
                print(f"{names[basis[i]]:>5} |", " ".join(f"{v:14.2f}" for v in T[i, :-1]), f"| {T[i,-1]:14.2f}")
            print(f"{'Z':>5} |", " ".join(f"{v:14.2f}" for v in T[m, :-1]), f"| {T[m,-1]:14.2f}")
        col = int(np.argmin(T[m, :-1]))
        if T[m, col] >= -1e-9:
            break
        ratios = [T[i, -1] / T[i, col] if T[i, col] > 1e-9 else np.inf for i in range(m)]
        row = int(np.argmin(ratios))
        if verbose:
            print(f"Kolom pivot: {names[col]}, baris pivot: {names[basis[row]]}, rasio = {ratios}")
        T[row] /= T[row, col]
        for i in range(m + 1):
            if i != row:
                T[i] -= T[i, col] * T[row]
        basis[row] = col
        it += 1
    x = np.zeros(n + m)
    for i, bi in enumerate(basis):
        x[bi] = T[i, -1]
    return x[:n], T[m, -1]


if __name__ == "__main__":
    #metode grafik
    print("=== Metode Grafik: titik sudut daerah layak ===")
    best = None
    for p in corner_points(A, b):
        z = c @ np.array(p)
        print(f"  ({p[0]:.0f}, {p[1]:.0f})  ->  Z = Rp {z:,.0f}")
        if best is None or z > best[1]:
            best = (p, z)
    print(f"Optimal (grafik): x1={best[0][0]:.0f}, x2={best[0][1]:.0f}, Z=Rp {best[1]:,.0f}")

    #simpleks manual
    print("\n=== Metode Simpleks (manual) ===")
    xs, zs = simplex_max(c, A, b)
    print(f"\nOptimal (simpleks): x1={xs[0]:.0f}, x2={xs[1]:.0f}, Z=Rp {zs:,.0f}")

    #scipy
    print("\n=== Verifikasi scipy.optimize.linprog ===")
    res = linprog(-c, A_ub=A, b_ub=b, bounds=[(0, None)] * 2, method="highs")
    print(f"Optimal (linprog): x1={res.x[0]:.0f}, x2={res.x[1]:.0f}, Z=Rp {-res.fun:,.0f}")
    print("Slack (sisa jam) senior/junior:", b - A @ res.x)

    #visualisasi
    x1 = np.linspace(0, 14, 400)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.plot(x1, (40 - 4 * x1) / 2, label="Senior: 4x1 + 2x2 = 40", color="#c0392b")
    ax.plot(x1, (36 - 2 * x1) / 3, label="Junior: 2x1 + 3x2 = 36", color="#2471a3")
    y_up = np.minimum((40 - 4 * x1) / 2, (36 - 2 * x1) / 3)
    ax.fill_between(x1, 0, np.maximum(y_up, 0), where=y_up >= 0, color="#aed6f1", alpha=0.5, label="Daerah layak")
    for p in corner_points(A, b):
        ax.plot(*p, "ko")
        ax.annotate(f"({p[0]:.0f}, {p[1]:.0f})", p, textcoords="offset points", xytext=(6, 6), fontsize=9)
    ax.plot(*best[0], "r*", markersize=18, label=f"Optimal (6, 8): Z = Rp {best[1]/1e6:.0f} juta")
    # garis isoprofit melalui titik optimal
    iso = (best[1] - 12e6 * x1) / 10e6
    ax.plot(x1, iso, "--", color="gray", label="Garis isoprofit Z optimal")
    ax.set_xlim(0, 14); ax.set_ylim(0, 22)
    ax.set_xlabel("x1 = paket Penetration Testing"); ax.set_ylabel("x2 = paket Security Audit")
    ax.set_title("Daerah Layak dan Solusi Optimal"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout(); fig.savefig("grafik_solusi.png", dpi=200)
    print("\nGrafik disimpan: grafik_solusi.png")


=== Metode Grafik: titik sudut daerah layak ===
  (0, 0)  ->  Z = Rp 0
  (0, 12)  ->  Z = Rp 120,000,000
  (6, 8)  ->  Z = Rp 152,000,000
  (10, 0)  ->  Z = Rp 120,000,000
Optimal (grafik): x1=6, x2=8, Z=Rp 152,000,000

=== Metode Simpleks (manual) ===

--- Tabel simpleks iterasi 0 ---
Basis |             x1             x2             s1             s2 |          RHS
   s1 |           4.00           2.00           1.00           0.00 |          40.00
   s2 |           2.00           3.00           0.00           1.00 |          36.00
    Z |   -12000000.00   -10000000.00           0.00           0.00 |           0.00
Kolom pivot: x1, baris pivot: s1, rasio = [np.float64(10.0), np.float64(18.0)]

--- Tabel simpleks iterasi 1 ---
Basis |             x1             x2             s1             s2 |          RHS
   x1 |           1.00           0.50           0.25           0.00 |          10.00
   s2 |           0.00           2.00          -0.50           1.00 |          16.00
    Z |  